## Next-Character Prediction (RNN)


In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 1. 데이터 준비 (Data Preparation)

# 1-1. 단어 리스트 정의 및 <eos> 토큰 추가
fruits = ["apple", "banana", "cherry", "orange", "grape", "mango", "peach", "melon", "kiwi", "lemon"]
data = [f"{word}<eos>" for word in fruits]

# 1-2. 문자 집합(Vocabulary) 생성
all_chars = sorted(list(set("".join(data))))
char_to_idx = {ch: i for i, ch in enumerate(all_chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(char_to_idx)

# 2. 학습 전 과일 이름 10가지 출력
print("--- 훈련 데이터 (10가지 과일) ---")
print(fruits)
print("-" * 35)
print(f"문자 집합 (Vocabulary): {''.join(all_chars)}")
print(f"단어집 크기 (특수문자 포함): {vocab_size}")
print("-" * 35)


# 3. RNN 모델 정의
class NextCharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(NextCharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        rnn_out, hidden = self.rnn(x, hidden)
        output = self.fc(rnn_out[-1]) 
        return output, hidden

    def init_hidden(self):
        return torch.zeros(self.num_layers, 1, self.hidden_size)

# 하이퍼파라미터 설정
input_size = vocab_size
hidden_size = 20
output_size = vocab_size
learning_rate = 0.005
num_layers = 1

# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")
print("-" * 35)

model = NextCharRNN(input_size, hidden_size, output_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


# 4. 모델 학습 (Training)

print("\n--- 모델 학습 시작 ---")
epochs = 300
for epoch in range(epochs):
    total_loss = 0
    for word in data:
        for i in range(1, len(word)):
            input_str = word[:i]
            label_char = word[i]
            
            input_indices = [char_to_idx[ch] for ch in input_str]
            label_index = char_to_idx[label_char]
            
            input_onehot = np.eye(vocab_size)[input_indices]
            
            inputs = torch.Tensor(input_onehot).view(len(input_str), 1, vocab_size).to(device)
            labels = torch.LongTensor([label_index]).to(device)
            
            hidden = model.init_hidden().to(device)
            optimizer.zero_grad()
            outputs, hidden = model(inputs, hidden)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Average Loss: {total_loss/len(data):.4f}")

print("-" * 35)


# 5. 사용자 입력 기반 예측 함수 정의 (★★ 수정된 버전 ★★)

def predict(model, start_str):
    """사용자가 입력한 시작 문자열로 다음 단어를 완성합니다."""
    model.eval() 
    with torch.no_grad():
        start_str = start_str.lower()
        device = next(model.parameters()).device
        
        # 시작 문자열을 텐서로 변환
        input_indices = [char_to_idx[ch] for ch in start_str]
        input_onehot = np.eye(vocab_size)[input_indices]
        inputs = torch.Tensor(input_onehot).view(len(start_str), 1, vocab_size).to(device)
        
        predicted_word = list(start_str)
        
        # --- 수정된 로직 시작 ---

        # 1. 시작 문자열을 모델에 통과시켜 첫 예측과 은닉 상태를 얻음
        hidden = model.init_hidden().to(device)
        initial_output, hidden = model(inputs, hidden)
        
        # 2. 첫 예측 결과를 단어에 추가
        _, top_idx = torch.max(initial_output, 1)
        predicted_char = idx_to_char[top_idx.item()]
        
        if predicted_char == '<eos>':
            return "".join(predicted_word)
        
        predicted_word.append(predicted_char)
        
        # 3. 방금 예측한 글자를 다음 루프의 첫 입력으로 사용
        last_char_idx = top_idx.item()
        
        # --- 수정된 로직 끝 ---

        # <eos>가 나오거나 최대 길이에 도달할 때까지 글자 생성 반복
        for _ in range(20): 
            input_onehot = np.eye(vocab_size)[[last_char_idx]]
            inputs = torch.Tensor(input_onehot).view(1, 1, vocab_size).to(device)
            
            # 이전 단계에서 업데이트된 hidden 상태를 그대로 사용
            output, hidden = model(inputs, hidden)
            
            _, top_idx = torch.max(output, 1)
            predicted_char = idx_to_char[top_idx.item()]
            
            if predicted_char == '<eos>':
                break
            
            predicted_word.append(predicted_char)
            last_char_idx = top_idx.item()

    return "".join(predicted_word)


# 6. 사용자 입력을 받아 예측 실행

print("\n--- 단어 완성기 ---")
print("단어의 시작 부분을 입력하세요. (예: 'ba', 'ch', 'le')")
print("종료하려면 'exit'를 입력하세요.")

while True:
    user_input = input("입력: ")
    if user_input.lower() == 'exit':
        print("프로그램을 종료합니다.")
        break
    
    try:
        completed_word = predict(model, user_input)
        print(f"  -> 모델 예측: {completed_word}")
    except KeyError:
        print(f"  -> 오류: 입력하신 '{user_input}'의 일부 글자가 단어집에 없습니다. 다른 글자로 시도해주세요.")
    except IndexError:
         print(f"  -> 오류: 입력이 비어있습니다. 글자를 입력해주세요.")

## Next-Character Prediction(LSTM)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 1. 데이터 준비 (이전과 동일)
fruits = ["apple", "banana", "cherry", "orange", "grape", "mango", "peach", "melon", "kiwi", "lemon"]
data = [f"{word}<eos>" for word in fruits]
all_chars = sorted(list(set("".join(data))))
char_to_idx = {ch: i for i, ch in enumerate(all_chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(char_to_idx)

print("--- 훈련 데이터 (10가지 과일) ---")
print(fruits)
print("-" * 35)

# 2. 모델 정의 (LSTM 모델)
class NextCharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(NextCharLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        lstm_out, hidden = self.lstm(x, hidden)
        output = self.fc(lstm_out[-1])
        return output, hidden

    def init_hidden(self):
        h0 = torch.zeros(self.num_layers, 1, self.hidden_size)
        c0 = torch.zeros(self.num_layers, 1, self.hidden_size)
        return (h0, c0)

# 하이퍼파라미터 설정
input_size = vocab_size
hidden_size = 40
output_size = vocab_size
learning_rate = 0.005
num_layers = 2

# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")
print("-" * 35)

model = NextCharLSTM(input_size, hidden_size, output_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


# 3. 모델 학습 (Training)
print("\n--- 모델 학습 시작 ---")
epochs = 300
for epoch in range(epochs):
    total_loss = 0
    for word in data:
        for i in range(1, len(word)):
            input_str = word[:i]
            label_char = word[i]
            
            input_indices = [char_to_idx[ch] for ch in input_str]
            label_index = char_to_idx[label_char]
            
            input_onehot = np.eye(vocab_size)[input_indices]
            
            inputs = torch.Tensor(input_onehot).view(len(input_str), 1, vocab_size).to(device)
            labels = torch.LongTensor([label_index]).to(device)
            
            hidden = tuple([h.to(device) for h in model.init_hidden()])
            
            optimizer.zero_grad()
            outputs, hidden = model(inputs, hidden)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Average Loss: {total_loss/len(data):.4f}")

print("-" * 35)


# 4. 사용자 입력 기반 예측 함수 정의 (★★ 최종 수정 버전 ★★)
def predict(model, start_str):
    model.eval()
    with torch.no_grad():
        start_str = start_str.lower()
        device = next(model.parameters()).device
        
        predicted_word = list(start_str)
        hidden = tuple([h.to(device) for h in model.init_hidden()])

        # 입력이 비어있는 경우, 시작 문자를 <eos>로 설정
        if not start_str:
            last_char_idx = char_to_idx['<eos>']
        else:
            # 1. 시작 문자열을 모델에 통과시켜 첫 예측과 문맥(hidden)을 얻음
            input_indices = [char_to_idx[ch] for ch in start_str]
            inputs = torch.Tensor(np.eye(vocab_size)[input_indices]).view(len(start_str), 1, vocab_size).to(device)
            output, hidden = model(inputs, hidden)
            
            # 2. 첫 예측 결과를 단어에 추가
            _, top_idx = torch.max(output, 1)
            predicted_char = idx_to_char[top_idx.item()]
            
            # 만약 첫 예측이 <eos>이면 바로 종료
            if predicted_char == '<eos>':
                return "".join(predicted_word)
            
            predicted_word.append(predicted_char)
            
            # 3. 방금 예측한 글자를 다음 생성 루프의 첫 입력으로 사용
            last_char_idx = top_idx.item()

        # 4. 나머지 단어를 생성하는 루프
        for _ in range(20):
            inputs = torch.Tensor(np.eye(vocab_size)[[last_char_idx]]).view(1, 1, vocab_size).to(device)
            output, hidden = model(inputs, hidden)
            
            _, top_idx = torch.max(output, 1)
            predicted_char = idx_to_char[top_idx.item()]
            
            if predicted_char == '<eos>':
                break
            
            predicted_word.append(predicted_char)
            last_char_idx = top_idx.item()

    return "".join(predicted_word)


# 5. 사용자 입력을 받아 예측 실행
print("\n--- 단어 완성기 ---")
print("단어의 시작 부분을 입력하세요. (예: 'ba', 'ch', 'le')")
print("종료하려면 'exit'를 입력하세요.")

while True:
    user_input = input("입력: ")
    if user_input.lower() == 'exit':
        print("프로그램을 종료합니다.")
        break
    
    try:
        completed_word = predict(model, user_input)
        print(f"  -> 모델 예측: {completed_word}")
    except KeyError as e:
        print(f"  -> 오류: '{e.args[0]}'는 단어집에 없는 글자입니다. 다른 글자로 시도해주세요.")
    except IndexError:
         print(f"  -> 오류: 입력이 비어있습니다. 글자를 입력해주세요.")